# 07 — Quantile Local Projections (Main Specification)

**Purpose:** Estimate the causal impulse response of DeFi liquidation shocks on ETH returns at each quantile of the return distribution. This replaces the static quantile regression AND the VAR-based IRF from the original draft with a single, more powerful framework.

**Method:**  Jordà (2005) local projections × Koenker & Bassett (1978) quantile regression.

For each horizon h ∈ {0, 1, ..., 24} and quantile τ ∈ {0.01, 0.05, 0.10, 0.50, 0.90, 0.95}:

$$Q_τ(R_{t→t+h}) = α(τ,h) + β(τ,h) · shock_t + δ(τ,h) · shock_t × OI\_high_t + γ(τ,h) · controls_t$$

where:
- R_{t→t+h} = cumulative return from t to t+h
- shock_t = log(1 + liq_usd_total_{t-1})  (lagged to reduce reverse causality)
- OI_high_t = 1 if OI > 80th percentile of rolling 30-day window
- controls = ret_btc_spot, vol_eth_7d, funding_rate, basis_bps

**Key coefficient:** β(τ,h) = response of quantile τ at horizon h to a liquidation shock.
The leverage cycle predicts: β(0.01, h) << 0 AND δ(0.01, h) << 0 (amplification when OI is high).

**Inference:** Newey-West HAC standard errors (12 lags) + block bootstrap (B=1000, block=24h).

**Note:** This notebook runs on the pre-DeFi panel with NaN liquidations as a DRY RUN to validate the pipeline. Once notebook 04 merges DeFi data, re-run with real data.

In [4]:
# ── Setup ──
import sys; sys.path.insert(0, "..")
from config import CFG, ECON_DIR

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Check for statsmodels
try:
    from statsmodels.regression.quantile_regression import QuantReg
    import statsmodels.api as sm
    print("statsmodels OK")
except ImportError:
    raise ImportError("pip install statsmodels")

print(f"Horizons : {CFG.ECON.lp_horizons}")
print(f"Quantiles: {CFG.ECON.quantiles}")

statsmodels OK
Horizons : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
Quantiles: [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]


In [6]:
# ── 1. Load panel ──

df = pd.read_parquet(CFG.FILES.econ_core_full, engine="pyarrow")
df["date"] = pd.to_datetime(df["date"], utc=True)
df = df.sort_values("date").reset_index(drop=True)

print(f"Panel: {len(df):,} rows × {df.shape[1]} cols")
print(f"DeFi data available: {df['liq_usd_total'].notna().any()}")

# ── For DRY RUN: simulate liquidation data ──
# if not df["liq_usd_total"].notna().any():
  #  print("\n⚠️  DRY RUN: generating synthetic liquidation data for pipeline testing")
   # np.random.seed(42)
    # Synthetic: liquidations spike when returns are very negative (mimics real mechanism)
   # liq_base = np.random.exponential(scale=5e4, size=len(df))
   # stress_mask = df["ret_eth_perp"].fillna(0) < df["ret_eth_perp"].quantile(0.05)
   # liq_base[stress_mask] *= 50  # amplify during stress
  #  df["liq_usd_total"] = liq_base
  #  df["log_liq"] = np.log1p(df["liq_usd_total"])
 #   print(f"  Synthetic liq_usd_total: mean={df['liq_usd_total'].mean():.0f}, max={df['liq_usd_total'].max():.0f}")

Panel: 41,328 rows × 26 cols
DeFi data available: True


In [8]:
# ── 2. Prepare LP variables ──

# Shock variable: LAGGED log-liquidation (t-1) to reduce reverse causality
df["shock"] = df["log_liq"].shift(1)

# Interaction: shock × OI regime
df["shock_x_oi_high"] = df["shock"] * df["oi_high"]

# Controls
controls = ["ret_btc_spot", "vol_eth_7d", "funding_rate", "basis_bps"]

# Cumulative forward returns for each horizon
for h in CFG.ECON.lp_horizons:
    if h == 0:
        df[f"cumret_h{h}"] = df["ret_eth_perp"]
    else:
        df[f"cumret_h{h}"] = df["ret_eth_perp"].rolling(h + 1).sum().shift(-h)

# Drop warm-up NaN rows
warmup = max(CFG.ECON.vol_window, 720) + max(CFG.ECON.lp_horizons) + 2
print(f"Warm-up rows dropped: {warmup}")
df_est = df.iloc[warmup:].copy()
print(f"Estimation sample: {len(df_est):,} rows")

Warm-up rows dropped: 746
Estimation sample: 40,582 rows


In [12]:
# ── 3. Quantile Local Projection estimation ──

regressors = ["shock", "shock_x_oi_high", "oi_high"] + controls
quantiles_main = [0.01, 0.05, 0.10, 0.50, 0.90, 0.95]

results = {}  # (tau, h) → dict of coefficients and SEs

for tau in quantiles_main:
    for h in CFG.ECON.lp_horizons:
        y_col = f"cumret_h{h}"
        
        # Build estimation matrix
        mask = df_est[[y_col, "shock"] + controls].notna().all(axis=1)
        y = df_est.loc[mask, y_col]
        X = sm.add_constant(df_est.loc[mask, regressors].fillna(0))
        
        if len(y) < 500:
            continue
        
        try:
            model = QuantReg(y, X)
            # Use Huber sandwich covariance (robust to heteroskedasticity)
            res = model.fit(q=tau, vcov="robust", kernel="epa", bandwidth="hsheather", max_iter=20000)
            
            results[(tau, h)] = {
                "beta_shock":      res.params.get("shock", np.nan),
                "se_shock":        res.bse.get("shock", np.nan),
                "pval_shock":      res.pvalues.get("shock", np.nan),
                "beta_interaction": res.params.get("shock_x_oi_high", np.nan),
                "se_interaction":  res.bse.get("shock_x_oi_high", np.nan),
                "pval_interaction": res.pvalues.get("shock_x_oi_high", np.nan),
                "n_obs":           int(res.nobs),
            }
        except Exception as e:
            print(f"  ⚠️ tau={tau}, h={h}: {e}")
    
    print(f"  τ = {tau:.2f} done ({len([k for k in results if k[0]==tau])} horizons)")

print(f"\nTotal estimates: {len(results)}")

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/regression/quantile_regression.py:191: IterationLimitWarning: Maximum number of iterations (20000) reached.
  warnings.warn("Maximum number of iterations (" + str(max_iter) +


  τ = 0.01 done (25 horizons)
  τ = 0.05 done (25 horizons)
  τ = 0.10 done (25 horizons)
  τ = 0.50 done (25 horizons)
  τ = 0.90 done (25 horizons)
  τ = 0.95 done (25 horizons)

Total estimates: 150


In [18]:
# ── 4. Reshape results into a table ──

rows = []
for (tau, h), v in sorted(results.items()):
    rows.append({"tau": tau, "h": h, **v})

df_res = pd.DataFrame(rows)

print("\n═══ KEY RESULTS: β(shock) by quantile and horizon ═══")
print("(negative β = liquidation shock depresses returns)\n")

pivot = df_res.pivot_table(index="h", columns="tau", values="beta_shock")
print(pivot.round(4).to_string())


═══ KEY RESULTS: β(shock) by quantile and horizon ═══
(negative β = liquidation shock depresses returns)

tau    0.01    0.05    0.10    0.50    0.90    0.95
h                                                  
0   -0.0322 -0.0194 -0.0134  0.0017  0.0143  0.0184
1   -0.0868 -0.0454 -0.0300  0.0056  0.0279  0.0307
2   -0.1207 -0.0498 -0.0296  0.0073  0.0368  0.0419
3   -0.1051 -0.0458 -0.0344  0.0100  0.0381  0.0439
4   -0.1596 -0.0591 -0.0386  0.0121  0.0458  0.0400
5   -0.1455 -0.0572 -0.0381  0.0125  0.0431  0.0532
6   -0.1594 -0.0792 -0.0431  0.0143  0.0467  0.0414
7   -0.1977 -0.0685 -0.0544  0.0151  0.0470  0.0507
8   -0.1896 -0.0708 -0.0509  0.0168  0.0496  0.0621
9   -0.2239 -0.0976 -0.0505  0.0203  0.0633  0.0714
10  -0.1979 -0.1098 -0.0569  0.0219  0.0674  0.0670
11  -0.1920 -0.1133 -0.0628  0.0257  0.0714  0.0822
12  -0.2411 -0.1017 -0.0606  0.0320  0.0699  0.0774
13  -0.2147 -0.1091 -0.0649  0.0291  0.0713  0.0835
14  -0.1975 -0.1097 -0.0603  0.0283  0.0754  0.0791
15  -0.20

In [20]:
# ── 5. Significance summary ──

print("\n═══ SIGNIFICANCE MAP (p < 0.05 marked with *) ═══\n")
sig_map = df_res.pivot_table(index="h", columns="tau", values="pval_shock")
sig_display = sig_map.map(lambda p: "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.10 else "")))
print(sig_display.to_string())

print("\n═══ INTERACTION β(shock × OI_high) at h=0 ═══")
for tau in quantiles_main:
    key = (tau, 0)
    if key in results:
        r = results[key]
        sig = "***" if r["pval_interaction"] < 0.01 else ("**" if r["pval_interaction"] < 0.05 else "")
        print(f"  τ={tau:.2f}: β_interact = {r['beta_interaction']:.4f}  (p={r['pval_interaction']:.4f}) {sig}")


═══ SIGNIFICANCE MAP (p < 0.05 marked with *) ═══

tau 0.01 0.05 0.10 0.50 0.90 0.95
h                                
0    ***  ***  ***  ***  ***  ***
1    ***  ***  ***  ***  ***  ***
2    ***  ***  ***  ***  ***  ***
3    ***  ***  ***  ***  ***  ***
4    ***  ***  ***  ***  ***  ***
5    ***  ***  ***  ***  ***  ***
6    ***  ***  ***  ***  ***  ***
7    ***  ***  ***  ***  ***  ***
8    ***  ***  ***  ***  ***  ***
9    ***  ***  ***  ***  ***  ***
10   ***  ***  ***  ***  ***  ***
11   ***  ***  ***  ***  ***  ***
12   ***  ***  ***  ***  ***  ***
13   ***  ***  ***  ***  ***  ***
14   ***  ***  ***  ***  ***  ***
15   ***  ***  ***  ***  ***  ***
16   ***  ***  ***  ***  ***  ***
17   ***  ***  ***  ***  ***  ***
18   ***  ***  ***  ***  ***  ***
19   ***  ***  ***  ***  ***  ***
20   ***  ***  ***  ***  ***  ***
21   ***  ***  ***  ***  ***  ***
22   ***  ***  ***  ***  ***  ***
23   ***  ***  ***  ***  ***  ***
24   ***  ***  ***  ***  ***  ***

═══ INTERACTION β(shock × OI_

In [15]:
# ── 6. Save results ──

from config import ECON_DIR
out_path = ECON_DIR / "quantile_lp_results.parquet"
df_res.to_parquet(out_path, index=False)
print(f"Saved: {out_path}")

# Also save as CSV for quick inspection
csv_path = ECON_DIR / "quantile_lp_results.csv"
df_res.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

# is_dry_run = "⚠️  DRY RUN — results are from SYNTHETIC data" if not df["log_liq"].isna().all() else ""
print("\n✅ Notebook 07 complete")

Saved: /Users/mirellaengerran/Desktop/Documents/Research_paper_leverage/data/econ/quantile_lp_results.parquet
Saved: /Users/mirellaengerran/Desktop/Documents/Research_paper_leverage/data/econ/quantile_lp_results.csv

✅ Notebook 07 complete


## Interpretation guide

**What to look for in the results:**

1. **β(shock) at τ=0.50** should be ≈ 0 → liquidations don't affect median returns (efficient market in normal times)

2. **β(shock) at τ=0.01** should be << 0 → liquidations drive returns deeper into the left tail (the "Liquidation Multiplier")

3. **β(interaction) at τ=0.01** should be << 0 → the multiplier is BIGGER when OI is high (leverage cycle amplification)

4. **Horizon profile:** β should peak at h=0 or h=1 then decay → the shock is absorbed within a few hours

5. **Pre-trend check:** if you add h=-1, h=-2 columns, β should be ≈ 0 (no anticipation → consistent with causal interpretation)

## Next steps

- **Notebook 08:** Block bootstrap for confidence bands + placebo test on non-collateral assets
- **Notebook 04:** Once DeFi data arrives, merge and re-run this notebook with real liquidation volumes
- **Figures:** Plot the IRF curves (β vs h) at each quantile for the paper